In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


BRONZE_SCHEMA = "football_data_platform.bronze"
SILVER_SCHEMA = "football_data_platform.silver"
SILVER_TABLE = f"{SILVER_SCHEMA}.team_standings_stats"

LEAGUES = ["bl", "ll", "lo", "pl", "sa"]


def clean_strings(df: DataFrame) -> DataFrame:
    """Trim string columns and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(F.trim(F.col(column)) == "", None)
                .otherwise(F.trim(F.col(column)))
            )

    return df


# ---------------------------------------------------------
# 1. Read Bronze tables
# ---------------------------------------------------------

dataframes = []

for league in LEAGUES:
    table_name = f"{BRONZE_SCHEMA}.fbref_standings_{league}"

    if not spark.catalog.tableExists(table_name):
        print(f"WARNING: Table does not exist: {table_name}")
        continue

    df = spark.table(table_name)
    df = clean_strings(df)

    dataframes.append(df)


if not dataframes:
    raise ValueError("No Bronze tables were found.")


# ---------------------------------------------------------
# 2. Combine Bronze data
# ---------------------------------------------------------

df_combined = dataframes[0]

for df in dataframes[1:]:
    df_combined = df_combined.unionByName(
        df,
        allowMissingColumns=True
    )


# ---------------------------------------------------------
# 3. Select and standardize columns
# ---------------------------------------------------------

df_silver = df_combined.select(
    # Business columns
    F.col("Squad").cast("string").alias("team_name"),
    F.col("W").cast("int").alias("matches_won"),
    F.col("D").cast("int").alias("matches_drawn"),
    F.col("L").cast("int").alias("matches_lost"),
    F.col("GF").cast("int").alias("goals_scored"),
    F.col("GA").cast("int").alias("goals_conceded"),
    F.col("GD").cast("int").alias("goal_difference"),
    F.col("Pts").cast("int").alias("points"),
    F.col("Pts/MP").cast("double").alias("points_per_match"),
    F.col("Attendance").cast("int").alias("average_attendance"),
    F.col("Top_Team_Scorer").cast("string").alias("top_scorer_name"),
    F.col("Goalkeeper").cast("string").alias("goalkeeper_name"),
  
    # Context
    F.col("_season").cast("string").alias("_season"),
    F.col("_league").cast("string").alias("_league"),

    # Bronze metadata
    F.col("_ingested_at").cast("timestamp").alias("_bronze_ingested_at"),
    F.col("_source_file").cast("string").alias("_bronze_source_file"),
    F.col("_source_system").cast("string").alias("_source_system"),
)


# ---------------------------------------------------------
# 4. Remove duplicates
# ---------------------------------------------------------

df_silver = df_silver.dropDuplicates(
    ["team_name", "_season", "_league"]
)


# ---------------------------------------------------------
# 5. Add Silver metadata
# ---------------------------------------------------------

df_silver = df_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)


# ---------------------------------------------------------
# 6. Basic validation
# ---------------------------------------------------------

null_team_names = df_silver.filter(
    F.col("team_name").isNull()
).count()

if null_team_names > 0:
    print(f"WARNING: NULL team_name: {null_team_names}")

print(f"Silver rows: {df_silver.count()}")


# ---------------------------------------------------------
# 7. Write Silver
# ---------------------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created {SILVER_TABLE}")